# Plot Volcano plots per condition

In [ ]:
%autosave 0

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import delnx as dx
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from dynaconf import Dynaconf
from tqdm import tqdm
from functools import reduce
import numpy as np
import pandas as pd
import marsilea as ma 
import marsilea.plotter as mp 
import scipy.cluster.hierarchy as sch

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Process DE results

In [ ]:
# Load DE results
de_results = pd.read_csv(Path(settings.ANALYSIS.tables_dir) / "annotation_de_results" / "tf_ko_panel_contrastiveVI_de_results_annotations.tsv", sep="\t")

In [ ]:
de_results.head()

In [ ]:
de_results.query("feature == 'FABP1' & annotation != 'Enterocytes' & padj < 0.05").sort_values("coef", ascending=False)

In [ ]:
# Rename x into group and condition into test_condition
de_results = de_results.rename(columns={"annotation": "group", "gene_name": "feature", "coef": "log2FoldChange"})

In [ ]:
# Add group_condition column
de_results["group_condition"] = de_results["group"] + "_" + de_results["condition"]

## Plot volcano plots for all comparisons

In [ ]:
import decoupler as dc

for group_cond in de_results["group_condition"].unique():
    de_df = de_results[de_results["group_condition"] == group_cond].copy()
    
    comparison_name = group_cond.replace(" ", "_").replace("/", "-")
    print(f"Generating volcano plot for comparison: {comparison_name}")
    
    df = de_df[['feature', 'log2FoldChange', 'padj']].copy()
    df = df.set_index('feature')
    df = df.dropna(subset=['padj', 'log2FoldChange'])
    
    df['log2FoldChange'] = df['log2FoldChange'].clip(lower=-10, upper=10)
    df["padj"] = df["padj"].clip(lower=np.exp(-300))
    
    fig = dc.pl.volcano(data=df, x="log2FoldChange", y="padj", figsize=(6,5), top=5, return_fig=True)
    fig.suptitle(f"Volcano Plot: {comparison_name}", fontsize=14)
    plt.grid(False)
    plt.tight_layout()
    
    outdir = Path(sc.settings.figdir) / "DE_11122025" / "per_condition_annotation_de_results_volcanos"
    outdir.mkdir(parents=True, exist_ok=True)
    
    plt.savefig(outdir / f"volcano_KO_{comparison_name}.pdf")
    plt.close()